In [ ]:
## Setup — packages & environment

import sys
import subprocess

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'networkx', 'openpyxl', 'reportlab']

for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os
import datetime as dt
from collections import defaultdict, Counter

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Event Log Creation

# Create synthetic event log
np.random.seed(RSEED)
activities = ['Access Course', 'View Lecture', 'Forum Discussion', 'Quiz Attempt', 'Submit Assignment', 'Receive Feedback']
n_cases = 35

events = []
for case_id in range(n_cases):
    # Typical learning path
    sequence = ['Access Course', 'View Lecture']
    # Randomly add middle activities
    if np.random.rand() > 0.3:
        sequence.append('Forum Discussion')
    if np.random.rand() > 0.2:
        sequence.append('Quiz Attempt')
    sequence.append('Submit Assignment')
    sequence.append('Receive Feedback')
    
    for idx, activity in enumerate(sequence):
        events.append({
            'case_id': f'case_{case_id}',
            'activity': activity,
            'timestamp': idx,
            'resource': np.random.choice(['System', 'Instructor', 'Peer'])
        })

log_df = pd.DataFrame(events)
print('Event Log:')
print(log_df.head(12))
print(f'\nTotal events: {len(log_df)}')
print(f'Unique activities: {log_df["activity"].nunique()}')

In [ ]:
## 2. Directly-Follows Graph

# Build directly-follows graph
dfg = defaultdict(Counter)

for case_id in log_df['case_id'].unique():
    case_events = log_df[log_df['case_id'] == case_id].sort_values('timestamp')
    activities_seq = case_events['activity'].tolist()
    
    for i in range(len(activities_seq) - 1):
        from_act = activities_seq[i]
        to_act = activities_seq[i + 1]
        dfg[from_act][to_act] += 1

print('Directly-Follows Graph (edge frequencies):')
for source in sorted(dfg.keys()):
    for target, count in dfg[source].items():
        print(f'{source} → {target}: {count}')

In [ ]:
## 3. Path Analysis

# Extract complete paths
paths = []
for case_id in log_df['case_id'].unique():
    case_events = log_df[log_df['case_id'] == case_id].sort_values('timestamp')
    path = ' → '.join(case_events['activity'].tolist())
    paths.append(path)

path_counts = Counter(paths)
print('Top 10 Most Frequent Paths:')
for path, count in path_counts.most_common(10):
    print(f'{count:2d}: {path}')

In [ ]:
## 4. NetworkX Graph Visualization

# Create directed graph
G = nx.DiGraph()

# Add nodes and edges with weights
for source in dfg:
    for target, weight in dfg[source].items():
        G.add_edge(source, target, weight=weight)

# Add nodes without outgoing edges
for activity in activities:
    if activity not in G:
        G.add_node(activity)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
## 5. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. DFG Network
fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50, seed=RSEED)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=3000, ax=ax)

# Draw edges with varying thickness based on weight
edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]
max_weight = max(weights) if weights else 1
edge_widths = [3 * w / max_weight for w in weights]

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6, ax=ax, 
                       edge_color='gray', arrowsize=20, arrowstyle='->')

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

# Draw edge weights
edge_labels = {(u, v): G[u][v]['weight'] for u, v in edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8, ax=ax)

ax.set_title('Directly-Follows Graph (Process Model)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/01_dfg_network.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_dfg_network.png')

# 2. Path frequency
fig, ax = plt.subplots(figsize=(10, 6))
top_paths = path_counts.most_common(8)
path_names = [f'Path {i+1}' for i in range(len(top_paths))]
frequencies = [count for _, count in top_paths]
colors = sns.color_palette('Set2', len(top_paths))
ax.barh(path_names, frequencies, color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 8 Most Frequent Process Paths', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(frequencies):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/02_path_frequencies.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_path_frequencies.png')

log_df.to_csv('process_log.csv', index=False)
print('\nSaved process log')

In [ ]:
## 6. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch14_ProcessMining_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 14: Process Mining</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Process mining discovers workflows by analyzing event logs. '
    'Directly-follows graphs reveal which activities typically follow one another, '
    'enabling identification of standard learning processes and deviations.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Methods</b>', styles['Heading2']))
methods = (
    'Event logs were analyzed to extract activity sequences per case. '
    'Directly-follows graph constructed by counting transitions. '
    'Paths ranked by frequency. Network visualization applied.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_dfg_network.png'):
        story.append(Image('figures/01_dfg_network.png', width=520, height=400))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Directly-follows graph showing activity transitions.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/02_path_frequencies.png'):
        story.append(Image('figures/02_path_frequencies.png', width=500, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Frequency of top learning paths.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>4. Interpretation</b>', styles['Heading2']))
interp = (
    'The process model reveals standardized learning workflows. '
    'Frequent paths indicate best practices; rare paths suggest alternative strategies. '
    'Bottlenecks appear as activities with high in-degree and low out-degree flow.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')